# 🚨 Benchmark modèles — Délinquance France par Région

## Stratégie

```
Fichier Excel (2012-2021)
   └── Zones (circumscriptions/compagnies)
         └── [Agrégation] → Départements
               └── [Agrégation] → Régions  ← SOURCE A
                                                  |
                                             Comparaison
                                                  |
data.gouv.fr CSV régional (2016-2025)  ← SOURCE B
```

**Benchmark de prédiction :**  
- Entraînement : 2012-2019 (Source A) ou 2016-2019 (Source B)  
- Test réel     : 2020-2021 (données observées disponibles dans les deux sources)  
- Modèles       : Prophet · XGBoost · LightGBM  
- Métrique      : RMSE + MAE + MAPE sur les années test

## Plan
1. Imports  
2. Source A — parsing Excel → département → région  
3. Source B — chargement data.gouv.fr régional  
4. Alignement & comparaison des deux sources  
5. EDA régionale  
6. Benchmark Prophet  
7. Benchmark XGBoost  
8. Benchmark LightGBM  
9. Tableau final — meilleur modèle par crime & région

## 1. Imports

In [ ]:
import subprocess, sys
for pkg in ['pandas','openpyxl','matplotlib','seaborn',
            'prophet','xgboost','lightgbm','scikit-learn']:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])
print('OK')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
plt.style.use('default')
sns.set_palette('husl')

# ── Métriques communes ──────────────────────────────────────────────────────
def rmse(y_true, y_pred):
    m = ~(np.isnan(y_true)|np.isnan(y_pred))
    return np.sqrt(np.mean((y_true[m]-y_pred[m])**2)) if m.sum()>=2 else np.nan

def mae(y_true, y_pred):
    m = ~(np.isnan(y_true)|np.isnan(y_pred))
    return np.mean(np.abs(y_true[m]-y_pred[m])) if m.sum()>=2 else np.nan

def mape(y_true, y_pred):
    m = ~(np.isnan(y_true)|np.isnan(y_pred)) & (y_true!=0)
    return np.mean(np.abs((y_true[m]-y_pred[m])/y_true[m]))*100 if m.sum()>=2 else np.nan

print('Imports OK')

## 2. Source A — Excel : Zone → Département → Région

In [ ]:
# ── Table correspondance Département → Région (codes INSEE 2016, préfixés 'R') ──
# Préfixe 'R' obligatoire : les codes régions (ex: 32, 84) collisionnent
# avec des codes département (32=Gers, 84=Vaucluse)
DEPT_REG = {
    # Auvergne-Rhône-Alpes
    '01':'R84','03':'R84','07':'R84','15':'R84','26':'R84','38':'R84',
    '42':'R84','43':'R84','63':'R84','69':'R84','73':'R84','74':'R84',
    # Bourgogne-Franche-Comté
    '21':'R27','25':'R27','39':'R27','58':'R27','70':'R27','71':'R27','89':'R27','90':'R27',
    # Bretagne
    '22':'R53','29':'R53','35':'R53','56':'R53',
    # Centre-Val de Loire
    '18':'R24','28':'R24','36':'R24','37':'R24','41':'R24','45':'R24',
    # Corse
    '2A':'R94','2B':'R94',
    # Grand Est
    '08':'R44','10':'R44','51':'R44','52':'R44','54':'R44','55':'R44',
    '57':'R44','67':'R44','68':'R44','88':'R44',
    # Hauts-de-France
    '02':'R32','59':'R32','60':'R32','62':'R32','80':'R32',
    # Île-de-France
    '75':'R11','77':'R11','78':'R11','91':'R11','92':'R11',
    '93':'R11','94':'R11','95':'R11',
    # Normandie
    '14':'R28','27':'R28','50':'R28','61':'R28','76':'R28',
    # Nouvelle-Aquitaine
    '16':'R75','17':'R75','19':'R75','23':'R75','24':'R75','33':'R75',
    '40':'R75','47':'R75','64':'R75','79':'R75','86':'R75','87':'R75',
    # Occitanie
    '09':'R76','11':'R76','12':'R76','30':'R76','31':'R76','32':'R76',
    '34':'R76','46':'R76','48':'R76','65':'R76','66':'R76','81':'R76','82':'R76',
    # Pays de la Loire
    '44':'R52','49':'R52','53':'R52','72':'R52','85':'R52',
    # PACA
    '04':'R93','05':'R93','06':'R93','13':'R93','83':'R93','84':'R93',
    # DROM
    '971':'R01','972':'R02','973':'R03','974':'R04','976':'R06',
    # COM (Polynésie, Nouvelle-Calédonie)
    '987':'R987','988':'R988',
}

REG_NAMES = {
    'R11':'Île-de-France',      'R24':'Centre-Val de Loire',
    'R27':'Bourgogne-FC',       'R28':'Normandie',
    'R32':'Hauts-de-France',    'R44':'Grand Est',
    'R52':'Pays de la Loire',   'R53':'Bretagne',
    'R75':'Nouvelle-Aquitaine', 'R76':'Occitanie',
    'R84':'Auvergne-RA',        'R93':'PACA',
    'R94':'Corse',
    'R01':'Guadeloupe','R02':'Martinique','R03':'Guyane',
    'R04':'La Réunion','R06':'Mayotte',
    'R987':'Polynésie','R988':'Nouvelle-Calédonie',
}
print('Table dept->région OK :', len(DEPT_REG), 'départements')

In [ ]:
# Adapter ce chemin si nécessaire
FICHIER = 'crimes-et-delits-enregistres-par-les-services-de-gendarmerie-et-de-police-depuis-2012.xlsx'

warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)
xl = pd.ExcelFile(FICHIER)
onglets = [s for s in xl.sheet_names if s != 'Présentation']
print(f'{len(onglets)} onglets')

In [ ]:
def parse_pn(xl, onglet):
    df   = pd.read_excel(xl, sheet_name=onglet, header=None).copy()
    annee = int(str(df.iloc[0,0]).split()[1])
    depts = df.iloc[0, 2:].values
    codes = df.iloc[3:, 0].values
    libs  = df.iloc[3:, 1].values
    vals  = df.iloc[3:, 2:].apply(pd.to_numeric, errors='coerce').values
    rows  = []
    for j, dept in enumerate(depts):
        d = str(dept).strip() if str(dept) not in ('nan','None') else None
        if not d: continue
        for i in range(len(codes)):
            rows.append({'annee':annee,'service':'PN','departement':d,
                         'code_index':codes[i],'libelle_crime':libs[i],'valeur':vals[i,j]})
    return pd.DataFrame(rows)

def parse_gn(xl, onglet):
    df   = pd.read_excel(xl, sheet_name=onglet, header=None).copy()
    annee = int(str(df.iloc[0,0]).split()[1])
    depts = df.iloc[0, 2:].values
    codes = df.iloc[2:, 0].values
    libs  = df.iloc[2:, 1].values
    vals  = df.iloc[2:, 2:].apply(pd.to_numeric, errors='coerce').values
    rows  = []
    for j, dept in enumerate(depts):
        d = str(dept).strip() if str(dept) not in ('nan','None') else None
        if not d: continue
        for i in range(len(codes)):
            rows.append({'annee':annee,'service':'GN','departement':d,
                         'code_index':codes[i],'libelle_crime':libs[i],'valeur':vals[i,j]})
    return pd.DataFrame(rows)

print('Parsing Excel (peut prendre ~1-2 min)...')
dfs = []
for onglet in onglets:
    dfs.append(parse_pn(xl,onglet) if 'PN' in onglet else parse_gn(xl,onglet))
    print(f'  {onglet}')

df_raw = pd.concat(dfs, ignore_index=True)
df_raw['valeur'] = pd.to_numeric(df_raw['valeur'], errors='coerce')
df_raw = df_raw.dropna(subset=['valeur'])

# Région
df_raw['region'] = df_raw['departement'].map(DEPT_REG)
df_raw['region_nom'] = df_raw['region'].map(REG_NAMES)

# Agrégation région (PN+GN, toutes zones)
df_A = (
    df_raw.dropna(subset=['region'])
    .groupby(['annee','region','region_nom','code_index','libelle_crime'])
    ['valeur'].sum().reset_index()
    .rename(columns={'valeur':'nb_faits_A'})
)

print(f'\nSource A : {df_A.shape[0]:,} lignes')
print(f'  Années   : {sorted(df_A["annee"].unique())}')
print(f'  Régions  : {df_A["region"].nunique()}')
print(f'  Crimes   : {df_A["code_index"].nunique()}')

## 3. Source B — data.gouv.fr CSV Régional

URL stable API data.gouv.fr : `2b27a675-e3bf-41ef-a852-5fb9ab483967`  
Période : 2016-2025 · Colonnes : `Code_region`, `annee`, `indicateur`, `nombre`, `taux_100k`

In [ ]:
URL_REG = 'https://www.data.gouv.fr/api/1/datasets/r/2b27a675-e3bf-41ef-a852-5fb9ab483967'

print('Chargement Source B (data.gouv.fr)...')
df_B_raw = pd.read_csv(URL_REG, sep=';', low_memory=False)
print(f'OK : {df_B_raw.shape}')
print('Colonnes :', df_B_raw.columns.tolist())
df_B_raw.head(3)

In [ ]:
# Nettoyage Source B
df_B = df_B_raw.copy()

# Renommage flexible
for src, dst in [('valeur_publiee','nombre'),('taux_pour_mille','taux_pm')]:
    if src in df_B.columns and dst not in df_B.columns:
        df_B = df_B.rename(columns={src: dst})

# Virgule -> point
for col in ['nombre','taux_pm','insee_pop']:
    if col in df_B.columns:
        df_B[col] = pd.to_numeric(
            df_B[col].astype(str).str.replace(',','.', regex=False), errors='coerce')

# Taux 100k
if 'taux_pm' in df_B.columns:
    df_B['taux_100k'] = df_B['taux_pm'] * 100
elif all(c in df_B.columns for c in ['nombre','insee_pop']):
    df_B['taux_100k'] = np.where(df_B['insee_pop']>0,
                                  df_B['nombre']/df_B['insee_pop']*100_000, np.nan)

# Code région -> préfixe 'R'
df_B['region'] = 'R' + df_B['Code_region'].astype(str).str.zfill(2)
df_B['region_nom'] = df_B['region'].map(REG_NAMES).fillna(df_B['region'])

df_B = df_B.dropna(subset=['nombre']).copy()

print(f'Source B nettoyée : {df_B.shape}')
print(f'  Années   : {sorted(df_B["annee"].unique())}')
print(f'  Régions  : {df_B["region"].nunique()}')
print(f'  Indicateurs : {df_B["indicateur"].nunique()}')
print('\nIndicateurs B :')
for ind in sorted(df_B['indicateur'].unique()):
    print(f'  {ind}')

## 4. Alignement des deux sources

La Source A utilise les **107 index historiques** (libellés longs).  
La Source B utilise des **indicateurs agrégés** (ex: "Cambriolages de logement").  
On aligne via une table de correspondance manuelle sur les indicateurs comparables.

In [ ]:
# Table de correspondance : indicateur Source B -> codes_index Source A à sommer
# Basée sur la documentation SSMSI
MAPPING_B_TO_A = {
    'Cambriolages de logement':                [27, 28],
    'Coups et blessures volontaires':          [44, 45],
    'Violences sexuelles':                     [46, 47],
    'Vols avec armes':                         [15, 16],
    "Vols violents sans arme":                 [17, 18],
    'Vols sans violence contre des personnes': [19, 20, 21, 22, 23, 24, 25, 26],
    'Destructions et dégradations volontaires':[63, 64],
    'Usage de stupéfiants':                    [91],
    'Trafic de stupéfiants':                   [92],
    'Escroqueries et abus de confiance':       [79],
    'Homicides':                               [1, 2, 3],
    "Tentatives d'homicides":                  [4, 5],
}

print('Indicateurs comparables :', len(MAPPING_B_TO_A))
for k, v in MAPPING_B_TO_A.items():
    print(f'  {k:<50} -> index {v}')

In [ ]:
# Construire df_A_mapped : agréger les codes A selon la table
rows_mapped = []
for indicateur_b, codes in MAPPING_B_TO_A.items():
    sub = df_A[df_A['code_index'].isin(codes)]
    agg = (
        sub.groupby(['annee','region','region_nom'])['nb_faits_A']
        .sum().reset_index()
    )
    agg['indicateur'] = indicateur_b
    rows_mapped.append(agg)

df_A_mapped = pd.concat(rows_mapped, ignore_index=True)

# Source B : garder uniquement les indicateurs mappés et années communes (2016-2021)
df_B_mapped = (
    df_B[df_B['indicateur'].isin(MAPPING_B_TO_A.keys())]
    [['annee','region','region_nom','indicateur','nombre']]
    .rename(columns={'nombre':'nb_faits_B'})
    .copy()
)

# Fusion sur années communes
df_compare = pd.merge(
    df_A_mapped,
    df_B_mapped,
    on=['annee','region','region_nom','indicateur'],
    how='inner'
)

# Ratio A/B : 1.0 = parfait alignement
df_compare['ratio_A_B'] = df_compare['nb_faits_A'] / df_compare['nb_faits_B'].replace(0, np.nan)

print(f'df_compare : {df_compare.shape}')
print(f'  Années communes   : {sorted(df_compare["annee"].unique())}')
print(f'  Régions communes  : {df_compare["region"].nunique()}')
print(f'  Indicateurs       : {df_compare["indicateur"].nunique()}')
print(f'\nRatio A/B moyen par indicateur (1.0 = sources identiques) :')
print(df_compare.groupby('indicateur')['ratio_A_B'].mean().round(3).to_string())

In [ ]:
# Visualisation de l'alignement A vs B
inds_plot = list(MAPPING_B_TO_A.keys())[:6]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, ind in enumerate(inds_plot):
    sub = df_compare[df_compare['indicateur'] == ind]
    nat_A = sub.groupby('annee')['nb_faits_A'].sum()
    nat_B = sub.groupby('annee')['nb_faits_B'].sum()

    axes[i].plot(nat_A.index, nat_A.values, 'o-', lw=2.5, ms=7,
                 color='steelblue', label='Source A (Excel)')
    axes[i].plot(nat_B.index, nat_B.values, 's--', lw=2.5, ms=7,
                 color='coral', label='Source B (data.gouv)')
    axes[i].set_title(ind[:40], fontweight='bold', fontsize=9)
    axes[i].set_xlabel('Année'); axes[i].set_ylabel('Nb faits')
    axes[i].legend(fontsize=7); axes[i].grid(True, alpha=0.3)

plt.suptitle('Alignement Source A (Excel) vs Source B (data.gouv.fr) — France entière',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('alignement_sources.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. EDA Régionale

In [ ]:
# Top 5 indicateurs pour les benchmarks
TOP_INDICATEURS = (
    df_compare.groupby('indicateur')['nb_faits_A'].sum()
    .sort_values(ascending=False).head(5).index.tolist()
)
print('Indicateurs retenus pour le benchmark :', TOP_INDICATEURS)

In [ ]:
# Heatmap régions × indicateurs (Source A, moyenne 2016-2021)
pivot_heat = (
    df_A_mapped[df_A_mapped['annee'].between(2016,2021)]
    .groupby(['region_nom','indicateur'])['nb_faits_A']
    .mean().unstack('indicateur').fillna(0)
)
# Normaliser par colonne pour lisibilité
pivot_norm = pivot_heat.div(pivot_heat.max())

plt.figure(figsize=(16, 9))
sns.heatmap(pivot_norm, cmap='YlOrRd', annot=False,
            cbar_kws={'label': 'Intensité relative (0-1)'})
plt.title('Profil criminologique régional — Source A (Excel, 2016-2021)',
          fontweight='bold', fontsize=13)
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('heatmap_regions.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Benchmark Prophet

**Protocole :** train sur 2012-2019 (Source A) + 2016-2019 (Source B),  
test sur **2020 et 2021** (années réelles observées dans les deux sources).

In [ ]:
from prophet import Prophet

TRAIN_END  = 2019
TEST_YEARS = [2020, 2021]

def prophet_predict(series_train, n_periods):
    """Entraîne Prophet et retourne les prédictions pour n_periods suivantes."""
    df_p = series_train.reset_index()
    df_p.columns = ['ds', 'y']
    df_p['ds'] = pd.to_datetime(df_p['ds'], format='%Y')
    m = Prophet(changepoint_prior_scale=0.05,
                yearly_seasonality=False, weekly_seasonality=False, daily_seasonality=False)
    m.fit(df_p)
    future = m.make_future_dataframe(periods=n_periods, freq='YS')
    fc = m.predict(future)
    preds = fc.set_index(fc['ds'].dt.year)['yhat']
    return preds

results_prophet = []

for ind in TOP_INDICATEURS:
    for region in df_compare['region'].unique():
        reg_nom = REG_NAMES.get(region, region)
        sub = df_compare[(df_compare['indicateur']==ind) & (df_compare['region']==region)]
        if len(sub) < 4: continue

        for source, col in [('A','nb_faits_A'), ('B','nb_faits_B')]:
            serie = sub.set_index('annee')[col].sort_index()
            train = serie[serie.index <= TRAIN_END]
            test  = serie[serie.index.isin(TEST_YEARS)]
            if len(train) < 4 or len(test) == 0: continue

            try:
                preds = prophet_predict(train, len(TEST_YEARS))
                y_true = test.values
                y_pred = np.array([preds.get(y, np.nan) for y in TEST_YEARS])

                results_prophet.append({
                    'modele':'Prophet','source':source,
                    'indicateur':ind,'region':region,'region_nom':reg_nom,
                    'RMSE': rmse(y_true, y_pred),
                    'MAE':  mae(y_true, y_pred),
                    'MAPE': mape(y_true, y_pred),
                })
            except Exception as e:
                pass

df_res_prophet = pd.DataFrame(results_prophet)
print(f'Prophet : {len(df_res_prophet)} évaluations')
print(df_res_prophet.groupby('source')[['RMSE','MAE','MAPE']].mean().round(2))

## 7. Benchmark XGBoost

In [ ]:
import xgboost as xgb

def make_features(serie):
    """Feature engineering sur une série temporelle annuelle."""
    df = serie.reset_index()
    df.columns = ['annee','y']
    df['year_sin']   = np.sin(2*np.pi*df['annee']/10)
    df['year_cos']   = np.cos(2*np.pi*df['annee']/10)
    df['year_trend'] = (df['annee']-df['annee'].min()) / max(1, df['annee'].max()-df['annee'].min())
    df['lag1']       = df['y'].shift(1).fillna(df['y'].mean())
    df['lag2']       = df['y'].shift(2).fillna(df['y'].mean())
    df['roll3']      = df['y'].rolling(3, min_periods=1).mean()
    return df

FEAT_COLS = ['year_sin','year_cos','year_trend','lag1','lag2','roll3']

results_xgb = []

for ind in TOP_INDICATEURS:
    for region in df_compare['region'].unique():
        reg_nom = REG_NAMES.get(region, region)
        sub = df_compare[(df_compare['indicateur']==ind) & (df_compare['region']==region)]
        if len(sub) < 4: continue

        for source, col in [('A','nb_faits_A'), ('B','nb_faits_B')]:
            serie = sub.set_index('annee')[col].sort_index()
            df_feat = make_features(serie).fillna(0)

            train_mask = df_feat['annee'] <= TRAIN_END
            test_mask  = df_feat['annee'].isin(TEST_YEARS)
            if train_mask.sum() < 4 or test_mask.sum() == 0: continue

            try:
                model = xgb.XGBRegressor(
                    n_estimators=50, max_depth=3, learning_rate=0.1,
                    random_state=42, verbosity=0, n_jobs=1)
                model.fit(df_feat.loc[train_mask, FEAT_COLS],
                          df_feat.loc[train_mask, 'y'])
                y_pred = model.predict(df_feat.loc[test_mask, FEAT_COLS])
                y_true = df_feat.loc[test_mask, 'y'].values

                results_xgb.append({
                    'modele':'XGBoost','source':source,
                    'indicateur':ind,'region':region,'region_nom':reg_nom,
                    'RMSE': rmse(y_true, y_pred),
                    'MAE':  mae(y_true, y_pred),
                    'MAPE': mape(y_true, y_pred),
                })
            except:
                pass

df_res_xgb = pd.DataFrame(results_xgb)
print(f'XGBoost : {len(df_res_xgb)} évaluations')
print(df_res_xgb.groupby('source')[['RMSE','MAE','MAPE']].mean().round(2))

## 8. Benchmark LightGBM

In [ ]:
import lightgbm as lgb

results_lgb = []

for ind in TOP_INDICATEURS:
    for region in df_compare['region'].unique():
        reg_nom = REG_NAMES.get(region, region)
        sub = df_compare[(df_compare['indicateur']==ind) & (df_compare['region']==region)]
        if len(sub) < 4: continue

        for source, col in [('A','nb_faits_A'), ('B','nb_faits_B')]:
            serie = sub.set_index('annee')[col].sort_index()
            df_feat = make_features(serie).fillna(0)

            train_mask = df_feat['annee'] <= TRAIN_END
            test_mask  = df_feat['annee'].isin(TEST_YEARS)
            if train_mask.sum() < 4 or test_mask.sum() == 0: continue

            try:
                model = lgb.LGBMRegressor(
                    n_estimators=50, max_depth=3, learning_rate=0.1,
                    num_leaves=15, random_state=42, verbose=-1, n_jobs=1)
                model.fit(df_feat.loc[train_mask, FEAT_COLS],
                          df_feat.loc[train_mask, 'y'])
                y_pred = model.predict(df_feat.loc[test_mask, FEAT_COLS])
                y_true = df_feat.loc[test_mask, 'y'].values

                results_lgb.append({
                    'modele':'LightGBM','source':source,
                    'indicateur':ind,'region':region,'region_nom':reg_nom,
                    'RMSE': rmse(y_true, y_pred),
                    'MAE':  mae(y_true, y_pred),
                    'MAPE': mape(y_true, y_pred),
                })
            except:
                pass

df_res_lgb = pd.DataFrame(results_lgb)
print(f'LightGBM : {len(df_res_lgb)} évaluations')
print(df_res_lgb.groupby('source')[['RMSE','MAE','MAPE']].mean().round(2))

## 9. Tableau final — Meilleur modèle par crime & région

In [ ]:
# Consolider tous les résultats
df_all_res = pd.concat([df_res_prophet, df_res_xgb, df_res_lgb], ignore_index=True)

# Résumé global
print('='*65)
print('RÉSUMÉ GLOBAL — MAPE moyen (% erreur) par modèle × source')
print('='*65)
pivot_summary = df_all_res.groupby(['modele','source'])['MAPE'].mean().round(1).unstack()
print(pivot_summary.to_string())
print()

print('RÉSUMÉ PAR INDICATEUR (MAPE moyen, source A + B confondues)')
print('-'*65)
pivot_ind = df_all_res.groupby(['modele','indicateur'])['MAPE'].mean().round(1).unstack('modele')
pivot_ind['Meilleur'] = pivot_ind.idxmin(axis=1)
print(pivot_ind.to_string())

In [ ]:
# Meilleur modèle par (indicateur, région, source)
df_best = (
    df_all_res
    .sort_values('MAPE')
    .groupby(['indicateur','region_nom','source'])
    .first()[['modele','RMSE','MAE','MAPE']]
    .reset_index()
)

# Comptage : combien de fois chaque modèle est le meilleur
wins = df_best['modele'].value_counts()
print('\n🏆 Classement — nombre de fois meilleur modèle :')
for model, count in wins.items():
    print(f'  {model:<12}: {count:3d} ({count/len(df_best)*100:.1f}%)')

In [ ]:
# ── Visualisations finales ────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. MAPE moyen par modèle
mape_model = df_all_res.groupby('modele')['MAPE'].mean().sort_values()
colors_bar = ['#2ecc71' if v == mape_model.min() else '#e74c3c'
              if v == mape_model.max() else '#3498db' for v in mape_model.values]
mape_model.plot(kind='bar', ax=axes[0,0], color=colors_bar, alpha=0.85)
axes[0,0].set_title('MAPE moyen par modèle (↓ = meilleur)',
                    fontweight='bold')
axes[0,0].set_ylabel('MAPE (%)'); axes[0,0].set_xticklabels(
    mape_model.index, rotation=0)
axes[0,0].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(mape_model.values):
    axes[0,0].text(i, v+0.2, f'{v:.1f}%', ha='center', fontweight='bold')

# 2. MAPE par modèle × source
pivot_src = df_all_res.groupby(['modele','source'])['MAPE'].mean().unstack()
pivot_src.plot(kind='bar', ax=axes[0,1], color=['steelblue','coral'], alpha=0.85)
axes[0,1].set_title('MAPE par modèle et source de données',
                    fontweight='bold')
axes[0,1].set_ylabel('MAPE (%)'); axes[0,1].set_xticklabels(
    pivot_src.index, rotation=0)
axes[0,1].legend(['Source A (Excel)','Source B (data.gouv)'])
axes[0,1].grid(True, alpha=0.3, axis='y')

# 3. Heatmap MAPE par modèle × indicateur
pivot_mi = df_all_res.groupby(['indicateur','modele'])['MAPE'].mean().unstack('modele')
pivot_mi.index = [x[:30] for x in pivot_mi.index]
sns.heatmap(pivot_mi, ax=axes[1,0], annot=True, fmt='.1f',
            cmap='RdYlGn_r', cbar_kws={'label': 'MAPE (%)'})
axes[1,0].set_title('MAPE par indicateur × modèle', fontweight='bold')
axes[1,0].set_xlabel('')
axes[1,0].tick_params(axis='x', rotation=30)

# 4. Nb victoires par modèle (pie)
axes[1,1].pie(wins.values, labels=wins.index, autopct='%1.1f%%',
              colors=['#2ecc71','#3498db','#e74c3c'][:len(wins)],
              startangle=90,
              wedgeprops={'edgecolor':'white','linewidth':2})
axes[1,1].set_title('Part des victoires (meilleur MAPE)',
                    fontweight='bold')

plt.suptitle('Benchmark Modèles — Prédiction Délinquance Régionale (test 2020-2021)',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('benchmark_final.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Visualisation prédictions vs réel — top indicateur, Île-de-France
ind_ex = TOP_INDICATEURS[0]
reg_ex = 'R11'   # Île-de-France (changer ici pour une autre région)
col_ex = 'nb_faits_A'

# Série annuelle : un seul point par année (somme si doublons)
sub_ex = (
    df_compare[
        (df_compare['indicateur'] == ind_ex) &
        (df_compare['region']     == reg_ex)
    ]
    .groupby('annee')[col_ex].sum()   # somme -> 1 valeur / année, index int
    .sort_index()
)
sub_ex.index = sub_ex.index.astype(int)

train_ex = sub_ex[sub_ex.index <= TRAIN_END]
test_ex  = sub_ex[sub_ex.index.isin(TEST_YEARS)]
y_true   = test_ex.reindex(TEST_YEARS).values  # shape (2,) garanti

print(f"Indicateur : {ind_ex}")
print(f"Région     : {REG_NAMES.get(reg_ex, reg_ex)}")
print(f"Train : {list(train_ex.index)}  ({len(train_ex)} pts)")
print(f"Test  : {list(test_ex.index)}  -> y_true={y_true}")

# ── Prophet ────────────────────────────────────────────────────────────────
preds_p  = prophet_predict(train_ex, len(TEST_YEARS))
y_pred_p = np.array([preds_p.get(y, np.nan) for y in TEST_YEARS])

# ── Features communes XGB + LGB ────────────────────────────────────────────
df_feat_ex = make_features(sub_ex).fillna(0).reset_index(drop=True)
df_feat_ex['annee'] = df_feat_ex['annee'].astype(int)
tm = df_feat_ex['annee'] <= TRAIN_END
te = df_feat_ex['annee'].isin(TEST_YEARS)

# ── XGBoost ────────────────────────────────────────────────────────────────
mxb = xgb.XGBRegressor(n_estimators=50, max_depth=3, learning_rate=0.1,
                        random_state=42, verbosity=0, n_jobs=1)
mxb.fit(df_feat_ex.loc[tm, FEAT_COLS], df_feat_ex.loc[tm, 'y'])
y_pred_x = mxb.predict(df_feat_ex.loc[te, FEAT_COLS])

# ── LightGBM ───────────────────────────────────────────────────────────────
mlb = lgb.LGBMRegressor(n_estimators=50, max_depth=3, learning_rate=0.1,
                         num_leaves=15, random_state=42, verbose=-1, n_jobs=1)
mlb.fit(df_feat_ex.loc[tm, FEAT_COLS], df_feat_ex.loc[tm, 'y'])
y_pred_l = mlb.predict(df_feat_ex.loc[te, FEAT_COLS])

# ── Graphique ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 6))

ax.plot(train_ex.index, train_ex.values, 'o-', color='black',
        lw=2, ms=7, label='Historique (train)')
ax.plot(TEST_YEARS, y_true,   'o',   color='black',
        ms=12, zorder=5, label='Réel 2020-2021')
ax.plot(TEST_YEARS, y_pred_p, 's--', color='coral',     lw=2, ms=9,
        label=f'Prophet  (MAPE={mape(y_true, y_pred_p):.1f}%)')
ax.plot(TEST_YEARS, y_pred_x, '^--', color='steelblue', lw=2, ms=9,
        label=f'XGBoost  (MAPE={mape(y_true, y_pred_x):.1f}%)')
ax.plot(TEST_YEARS, y_pred_l, 'D--', color='seagreen',  lw=2, ms=9,
        label=f'LightGBM (MAPE={mape(y_true, y_pred_l):.1f}%)')

ax.axvline(TRAIN_END + 0.5, color='gray', ls=':', alpha=0.7)
ax.text(TRAIN_END + 0.6, ax.get_ylim()[1] * 0.95, 'Test ->', color='gray', fontsize=10)
ax.set_title(
    f'{ind_ex[:55]}\n{REG_NAMES.get(reg_ex, reg_ex)} — Source A'
    f' (train<={TRAIN_END}, test {TEST_YEARS})',
    fontweight='bold')
ax.set_xlabel('Annee'); ax.set_ylabel('Nb faits')
ax.legend(loc='upper left'); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('exemple_predictions.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Export Excel complet ──────────────────────────────────────────────────────
with pd.ExcelWriter('benchmark_delinquance_regional.xlsx', engine='openpyxl') as w:
    df_all_res.to_excel(w, sheet_name='Tous_resultats', index=False)
    df_best.to_excel(w, sheet_name='Meilleur_par_cas', index=False)
    pivot_summary.to_excel(w, sheet_name='Resume_MAPE_modele_source')
    pivot_ind.to_excel(w, sheet_name='MAPE_par_indicateur')
    df_compare.to_excel(w, sheet_name='Comparaison_A_vs_B', index=False)

print('Exporté : benchmark_delinquance_regional.xlsx')

print('\n' + '='*65)
print('RAPPORT FINAL — BENCHMARK MODÈLES')
print('='*65)
print(f'Indicateurs testés : {len(TOP_INDICATEURS)}')
print(f'Régions testées    : {df_compare["region"].nunique()}')
print(f'Années de test     : {TEST_YEARS}')
print()
print('MAPE moyen global :')
for model in ['Prophet','XGBoost','LightGBM']:
    sub = df_all_res[df_all_res['modele']==model]
    print(f'  {model:<12}: {sub["MAPE"].mean():.1f}%  '
          f'(RMSE={sub["RMSE"].mean():.0f})')
print()
print('Meilleur modèle global :', df_all_res.groupby('modele')['MAPE'].mean().idxmin())
print('='*65)